# Phase 6 PLAN 3 — Prototype Loss Replacement
Eight preregistered ablations with frozen BatchNorm, one target pass per epoch, a single target forward, and protected epoch-0 initialization. Source-validation Macro-F1 is the only checkpoint-selection signal; target metrics are post-training descriptions.

## 1. Setup

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/CHANGE_ME/Best-thesis-in-council.git'
BRANCH = 'main'
RUN_FULL = False
DOMAIN_PAIR = 'ds1_ds2'  # ds1_ds2, ds1_incart, ds1_svdb, mitbih_incart, mitbih_svdb
TRAIN_BASE_IF_MISSING = False
ENABLE_WANDB = False
WANDB_PROJECT = 'ecg-thesis'
WANDB_ENTITY = None
WANDB_GROUP = 'phase6-daeac-prototype-loss'
RESUME_CHECKPOINTS = {}  # {'full_pair': '/kaggle/input/.../daeac_proto_loss_full_pair_latest.pt'}
DATA_INPUT_DIR = None  # optionally pin one mounted directory
BASE_CHECKPOINT_INPUT = None  # optionally pin daeac_base_best.pt

PAIR_CONFIGS = {name: f'configs/phase6_daeac_pair_{name}.yaml' for name in ('ds1_ds2', 'ds1_incart', 'ds1_svdb', 'mitbih_incart', 'mitbih_svdb')}
assert DOMAIN_PAIR in PAIR_CONFIGS, DOMAIN_PAIR
VARIANTS = ['full_pair']  # Domain-pair runs use the paper-faithful full loss.
WORK = Path('/kaggle/working')
ROOT = WORK / 'Best-thesis-in-council'
REPO = ROOT / 'ecg_thesis'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, str(ROOT)], check=True)
os.chdir(REPO)
if ENABLE_WANDB:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
def run(args):
    print('+', ' '.join(str(x) for x in args), flush=True)
    subprocess.run([str(x) for x in args], cwd=REPO, check=True)
def config_for(variant):
    return PAIR_CONFIGS[DOMAIN_PAIR]
print('Repository:', REPO)


## 2. Dependencies

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

## 3. Copy immutable Kaggle inputs into `/kaggle/working`

In [ ]:
INPUT = Path('/kaggle/input')
DATA_DEST = REPO / 'data/processed/phase6_daeac_paper'
CKPT_DEST = REPO / 'outputs/phase6_daeac_paper/checkpoints'
DATA_DEST.mkdir(parents=True, exist_ok=True)
CKPT_DEST.mkdir(parents=True, exist_ok=True)
PAIR_DATA = {
    'ds1_ds2': ['mitdb_ds1_daeac.npz', 'mitdb_ds2_daeac.npz'],
    'ds1_incart': ['mitdb_ds1_daeac.npz', 'incart_all_daeac.npz'],
    'ds1_svdb': ['mitdb_ds1_daeac.npz', 'svdb_all_daeac.npz'],
    'mitbih_incart': ['mitdb_all_daeac.npz', 'incart_all_daeac.npz'],
    'mitbih_svdb': ['mitdb_all_daeac.npz', 'svdb_all_daeac.npz'],
}
required_data = PAIR_DATA[DOMAIN_PAIR]
optional_data = []
def unique_input(name, pinned=None, required=True):
    if pinned:
        path = Path(pinned)
        if not path.exists(): raise FileNotFoundError(path)
        return path
    root = Path(DATA_INPUT_DIR) if DATA_INPUT_DIR else INPUT
    matches = list(root.rglob(name))
    if len(matches) > 1: raise RuntimeError(f'Multiple {name} inputs; pin its path: {matches}')
    if not matches:
        if required: raise FileNotFoundError(f'Missing Kaggle input: {name}')
        return None
    return matches[0]
for name in required_data + optional_data:
    source = unique_input(name, required=name in required_data)
    if source: shutil.copy2(source, DATA_DEST / name)
PAIR_OUTPUT = REPO / f'outputs/phase6_daeac_pair_{DOMAIN_PAIR}'
base_prefix = 'daeac_base_mitbih_all' if DOMAIN_PAIR.startswith('mitbih_') else 'daeac_base_ds1'
base_dest = PAIR_OUTPUT / 'checkpoints' / f'{base_prefix}_best.pt'
base = unique_input(f'{base_prefix}_best.pt', BASE_CHECKPOINT_INPUT, required=not TRAIN_BASE_IF_MISSING)
if base:
    base_dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(base, base_dest)
print('Inputs copied; /kaggle/input remains untouched.')


## 4. Static, unit, after5, and strict protocol checks

In [ ]:
run([sys.executable, 'scripts/check_repo.py'])
run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_daeac_prototype*.py'])
run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_daeac_pseudo_filter.py'])
run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_daeac_losses.py'])
run([sys.executable, 'scripts/phase6_daeac_proto_loss/00_prepare_after5.py', '--config', config_for('full_pair')])
if not base_dest.exists():
    run([sys.executable, 'scripts/phase6_daeac_paper/01_train_base.py', '--config', config_for('full_pair')])
for variant in VARIANTS:
    run([sys.executable, 'scripts/phase6_daeac_proto_loss/01_validate.py', '--config', config_for(variant), '--strict'])


## 5. One-epoch smoke tests (not experiment evidence)

In [ ]:
for variant in VARIANTS:
    run([sys.executable, 'scripts/phase6_daeac_proto_loss/02_train.py', '--config', config_for(variant), '--epochs', '1', '--max-source-samples', '1024', '--max-target-samples', '1024', '--max-val-samples', '512', '--output-dir', f'/kaggle/working/smoke/proto_loss_{variant}', '--checkpoint-prefix', f'proto_loss_{variant}_smoke'])
print(f'Paper-faithful smoke test passed for {DOMAIN_PAIR}.')


## 6. Full 300-epoch domain-pair run (explicit opt-in)

In [ ]:
if RUN_FULL:
    for variant in VARIANTS:
        cmd = [sys.executable, 'scripts/phase6_daeac_proto_loss/02_train.py', '--config', config_for(variant)]
        if variant in RESUME_CHECKPOINTS: cmd += ['--resume-checkpoint', RESUME_CHECKPOINTS[variant]]
        if ENABLE_WANDB:
            cmd += ['--wandb', '--wandb-project', WANDB_PROJECT, '--wandb-group', WANDB_GROUP, '--wandb-run-name', f'proto-loss-{variant}']
            if WANDB_ENTITY: cmd += ['--wandb-entity', WANDB_ENTITY]
        run(cmd)
else:
    print(f'Full 300-epoch {DOMAIN_PAIR} run skipped. Set RUN_FULL=True after the smoke check passes.')


## 7. Post-training held-out evaluation

In [ ]:
if RUN_FULL:
    for variant in VARIANTS:
        prefix = f'daeac_{DOMAIN_PAIR}'
        checkpoint = PAIR_OUTPUT / 'checkpoints' / f'{prefix}_best.pt'
        if not checkpoint.exists(): raise FileNotFoundError(checkpoint)
        run([sys.executable, 'scripts/phase6_daeac_proto_loss/03_eval.py', '--config', config_for(variant), '--checkpoint', checkpoint, '--dataset', 'all'])
    REPORT = PAIR_OUTPUT / 'metrics'
    print('Pair metrics:', REPORT)


## 8. Persist outputs

In [ ]:
if RUN_FULL:
    BUNDLE = WORK / 'phase6_daeac_proto_loss_bundle'
    if BUNDLE.exists(): shutil.rmtree(BUNDLE)
    BUNDLE.mkdir(parents=True)
    for variant in VARIANTS:
        source = PAIR_OUTPUT
        shutil.copytree(source, BUNDLE / source.name)
    archive = shutil.make_archive(str(WORK / 'phase6_daeac_proto_loss_bundle'), 'zip', BUNDLE)
    print('Saved:', archive)
